In [1]:
# Step 1: Calculate overall campaign performance
import pandas as pd 
import os 
import plotly.express as px 
import plotly.graph_objects as go 
from plotly.subplots import make_subplots

# Set working directory 
os.chdir(r'D:\Mission_Blitzkreig\12.Load_clean_calculate_metrics')

# Load data 
df = pd.read_csv('ooh_campaigns.csv')

# Calculate metrics 
df['ctr'] = (df['clicks'] / df['impressions'] * 100).round(2)
df['cpm'] = (df['spend'] / (df['impressions'] / 1000)).round(2)
df['roi'] = ((df['revenue'] - df['spend']) / df['spend'] * 100).round(2)
df['cvr'] = (df['conversions'] / df['clicks'] * 100).round(2)
df['profit'] = df['revenue'] - df['spend']

# Clean data
df = df.fillna(0)
df = df.dropna(subset=['impressions', 'spend', 'revenue'])

print("✅ Data loaded and cleaned!")

✅ Data loaded and cleaned!


In [5]:
# Calculate Summary Stats 
summary_stats = {
    'Total Campaigns': len(df),
    'Total Spend': f"€{df['spend'].sum():,.2f}",
    'Total Revenue': f"€{df['revenue'].sum():,.2f}",
    'Total Profit': f"€{df['profit'].sum():,.2f}",
    'Average ROI': f"{df['roi'].mean():.1f}%",
    'Average CTR': f"{df['ctr'].mean():.2f}%",
    'Average CPM': f"€{df['cpm'].mean():.2f}",
    'Total Impressions': f"{df['impressions'].sum():,.0f}",
    'Total Clicks': f"{df['clicks'].sum():,.0f}",
    'Total Conversions': f"{df['conversions'].sum():,.0f}",
    'Best Performing City': df.groupby('city')['roi'].mean().idxmax(),
    'Best Performing Format': df.groupby('format')['roi'].mean().idxmax()
}

print("\n📊 CAMPAIGN SUMMARY STATISTICS:")
for key, value in summary_stats.items():
    print(f"{key}: {value}")



📊 CAMPAIGN SUMMARY STATISTICS:
Total Campaigns: 10
Total Spend: €33,195.00
Total Revenue: €199,700.00
Total Profit: €166,505.00
Average ROI: 592.5%
Average CTR: 3.12%
Average CPM: €97.99
Total Impressions: 5,113,456
Total Clicks: 11,150
Total Conversions: 747
Best Performing City: Belfast
Best Performing Format: 6 Sheet


In [7]:
# Create 3x3 subplot grid
fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=(
        '💰 Revenue by City',
        '📊 Spend vs Revenue',
        '🎯 ROI by Format',
        '📈 CTR vs ROI by City',
        '🏆 Top 5 Campaigns',
        '📍 Conversions by City',
        '🎨 Revenue by Format',
        '📉 Performance Metrics',
        '💡 Key Insights'
    ),
    specs=[
        [{'type': 'bar'}, {'type': 'scatter'}, {'type': 'bar'}],
        [{'type': 'scatter'}, {'type': 'bar'}, {'type': 'bar'}],
        [{'type': 'pie'}, {'type': 'table'}, {'type': 'table'}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.10
)

# ===== ROW 1, COL 1: Revenue by City =====
city_revenue = df.groupby('city')['revenue'].sum().reset_index().sort_values('revenue', ascending=False)

fig.add_trace(
    go.Bar(
        x=city_revenue['city'],
        y=city_revenue['revenue'],
        name='Revenue',
        marker_color='steelblue'
    ),
    row=1, col=1
)

# ===== ROW 1, COL 2: Spend vs Revenue Scatter =====
fig.add_trace(
    go.Scatter(
        x=df['spend'],
        y=df['revenue'],
        mode='markers',
        marker=dict(
            size=df['impressions']/50000,
            color=df['roi'],
            colorscale='RdYlGn',
            showscale=True,
            colorbar=dict(title="ROI %", x=0.65)
        ),
        text=df['campaign_name'],
        name='Campaigns'
    ),
    row=1, col=2
)

# Add break-even line
max_val = max(df['spend'].max(), df['revenue'].max())
fig.add_trace(
    go.Scatter(
        x=[0, max_val],
        y=[0, max_val],
        mode='lines',
        line=dict(color='red', dash='dash'),
        name='Break-even',
        showlegend=False
    ),
    row=1, col=2
)

# ===== ROW 1, COL 3: ROI by Format =====
format_roi = df.groupby('format')['roi'].mean().reset_index().sort_values('roi', ascending=False)

fig.add_trace(
    go.Bar(
        x=format_roi['format'],
        y=format_roi['roi'],
        name='Avg ROI',
        marker_color='lightgreen',
        text=format_roi['roi'].round(1),
        texttemplate='%{text}%',
        textposition='outside'
    ),
    row=1, col=3
)

# ===== ROW 2, COL 1: CTR vs ROI by City =====
for city in df['city'].unique():
    city_data = df[df['city'] == city]
    fig.add_trace(
        go.Scatter(
            x=city_data['ctr'],
            y=city_data['roi'],
            mode='markers',
            name=city,
            marker=dict(size=8),
            showlegend=True
        ),
        row=2, col=1
    )

# ===== ROW 2, COL 2: Top 5 Campaigns =====
top_5 = df.nlargest(5, 'roi')[['campaign_name', 'roi']]
# Shorten campaign names for display
top_5['short_name'] = top_5['campaign_name'].str[:30]

fig.add_trace(
    go.Bar(
        x=top_5['roi'],
        y=top_5['short_name'],
        orientation='h',
        name='Top ROI',
        marker_color='gold',
        text=top_5['roi'].round(1),
        texttemplate='%{text}%',
        textposition='outside'
    ),
    row=2, col=2
)

# ===== ROW 2, COL 3: Conversions by City =====
conversions_city = df.groupby('city')['conversions'].sum().reset_index().sort_values('conversions', ascending=False)

fig.add_trace(
    go.Bar(
        x=conversions_city['city'],
        y=conversions_city['conversions'],
        name='Conversions',
        marker_color='coral'
    ),
    row=2, col=3
)

# ===== ROW 3, COL 1: Revenue by Format (Pie) =====
format_revenue = df.groupby('format')['revenue'].sum().reset_index()

fig.add_trace(
    go.Pie(
        labels=format_revenue['format'],
        values=format_revenue['revenue'],
        name='Format Revenue'
    ),
    row=3, col=1
)

# ===== ROW 3, COL 2: Performance Metrics Table =====
metrics_table = pd.DataFrame({
    'Metric': ['Total Spend', 'Total Revenue', 'Total Profit', 'Avg ROI', 'Avg CTR', 'Avg CPM'],
    'Value': [
        f"€{df['spend'].sum():,.0f}",
        f"€{df['revenue'].sum():,.0f}",
        f"€{df['profit'].sum():,.0f}",
        f"{df['roi'].mean():.1f}%",
        f"{df['ctr'].mean():.2f}%",
        f"€{df['cpm'].mean():.2f}"
    ]
})

fig.add_trace(
    go.Table(
        header=dict(
            values=['<b>Metric</b>', '<b>Value</b>'],
            fill_color='steelblue',
            font=dict(color='white', size=14),
            align='left'
        ),
        cells=dict(
            values=[metrics_table['Metric'], metrics_table['Value']],
            fill_color='lavender',
            font=dict(size=12),
            align='left',
            height=30
        )
    ),
    row=3, col=2
)

# ===== ROW 3, COL 3: Key Insights Table =====
insights = pd.DataFrame({
    'Insight': [
        'Best City',
        'Best Format',
        'Total Campaigns',
        'Total Impressions',
        'Total Clicks',
        'Total Conversions'
    ],
    'Detail': [
        df.groupby('city')['roi'].mean().idxmax(),
        df.groupby('format')['roi'].mean().idxmax(),
        str(len(df)),
        f"{df['impressions'].sum():,.0f}",
        f"{df['clicks'].sum():,.0f}",
        f"{df['conversions'].sum():,.0f}"
    ]
})

fig.add_trace(
    go.Table(
        header=dict(
            values=['<b>Insight</b>', '<b>Detail</b>'],
            fill_color='darkgreen',
            font=dict(color='white', size=14),
            align='left'
        ),
        cells=dict(
            values=[insights['Insight'], insights['Detail']],
            fill_color='lightgreen',
            font=dict(size=12),
            align='left',
            height=30
        )
    ),
    row=3, col=3
)

# ===== Update Layout =====
fig.update_layout(
    title_text="<b>OOH Campaign Performance Dashboard - Complete Analysis</b>",
    title_font_size=24,
    showlegend=True,
    height=1200,
    width=1800,
    font=dict(family="Arial", size=11)
)

# Update axes
fig.update_xaxes(title_text="City", row=1, col=1)
fig.update_yaxes(title_text="Revenue (€)", row=1, col=1)
fig.update_xaxes(title_text="Spend (€)", row=1, col=2)
fig.update_yaxes(title_text="Revenue (€)", row=1, col=2)
fig.update_xaxes(title_text="Format", row=1, col=3)
fig.update_yaxes(title_text="ROI (%)", row=1, col=3)
fig.update_xaxes(title_text="CTR (%)", row=2, col=1)
fig.update_yaxes(title_text="ROI (%)", row=2, col=1)
fig.update_xaxes(title_text="ROI (%)", row=2, col=2)
fig.update_xaxes(title_text="City", row=2, col=3)
fig.update_yaxes(title_text="Conversions", row=2, col=3)

# Show and save
fig.show()
fig.write_html('FINAL_DASHBOARD.html')
print("\n🎉 ✅ FINAL DASHBOARD SAVED: FINAL_DASHBOARD.html")


🎉 ✅ FINAL DASHBOARD SAVED: FINAL_DASHBOARD.html


In [8]:
# Create executive summary
executive_summary = f"""
{'='*80}
OOH CAMPAIGN PERFORMANCE REPORT
{'='*80}

EXECUTIVE SUMMARY
-----------------
Total Campaigns Analyzed: {len(df)}
Campaign Period: {df['campaign_name'].iloc[0]} to {df['campaign_name'].iloc[-1]}

FINANCIAL PERFORMANCE
---------------------
Total Investment: €{df['spend'].sum():,.2f}
Total Revenue Generated: €{df['revenue'].sum():,.2f}
Total Profit: €{df['profit'].sum():,.2f}
Overall ROI: {((df['revenue'].sum() - df['spend'].sum()) / df['spend'].sum() * 100):.1f}%

ENGAGEMENT METRICS
------------------
Total Impressions: {df['impressions'].sum():,.0f}
Total Clicks: {df['clicks'].sum():,.0f}
Total Conversions: {df['conversions'].sum():,.0f}
Average CTR: {df['ctr'].mean():.2f}%
Average CPM: €{df['cpm'].mean():.2f}
Conversion Rate: {(df['conversions'].sum() / df['clicks'].sum() * 100):.2f}%

TOP PERFORMERS
--------------
Best City (by ROI): {df.groupby('city')['roi'].mean().idxmax()} ({df.groupby('city')['roi'].mean().max():.1f}% ROI)
Best Format (by ROI): {df.groupby('format')['roi'].mean().idxmax()} ({df.groupby('format')['roi'].mean().max():.1f}% ROI)

Top 3 Campaigns:
"""

top_3 = df.nlargest(3, 'roi')[['campaign_name', 'roi', 'revenue']]
for idx, row in top_3.iterrows():
    executive_summary += f"  {idx+1}. {row['campaign_name'][:50]} - ROI: {row['roi']:.1f}%, Revenue: €{row['revenue']:,.2f}\n"

executive_summary += f"""

RECOMMENDATIONS
---------------
1. Focus on {df.groupby('format')['roi'].mean().idxmax()} format - highest ROI
2. Expand campaigns in {df.groupby('city')['roi'].mean().idxmax()} - best performing city
3. Review underperforming campaigns (ROI < 50%) for optimization

{'='*80}
"""

print(executive_summary)

# Save to file
with open('Executive_Summary.txt', 'w') as f:
    f.write(executive_summary)

print("✅ Executive summary saved: Executive_Summary.txt")


OOH CAMPAIGN PERFORMANCE REPORT

EXECUTIVE SUMMARY
-----------------
Total Campaigns Analyzed: 10
Campaign Period: Dublin City Centre - 48 Sheet to Waterford Retail Park - 6 Sheet

FINANCIAL PERFORMANCE
---------------------
Total Investment: €33,195.00
Total Revenue Generated: €199,700.00
Total Profit: €166,505.00
Overall ROI: 501.6%

ENGAGEMENT METRICS
------------------
Total Impressions: 5,113,456
Total Clicks: 11,150
Total Conversions: 747
Average CTR: 3.12%
Average CPM: €97.99
Conversion Rate: 6.70%

TOP PERFORMERS
--------------
Best City (by ROI): Belfast (1666.1% ROI)
Best Format (by ROI): 6 Sheet (1570.8% ROI)

Top 3 Campaigns:
  3. Belfast Roadside - 6 Sheet - ROI: 2900.0%, Revenue: €45,000.00
  4. Dublin Airport - Supersite - ROI: 464.7%, Revenue: €48,000.00
  9. Belfast City Hall - Supersite - ROI: 432.3%, Revenue: €33,000.00


RECOMMENDATIONS
---------------
1. Focus on 6 Sheet format - highest ROI
2. Expand campaigns in Belfast - best performing city
3. Review underperf